<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/07)_%EB%8B%B5%EB%B3%80_%EA%B8%B0%EB%8A%A5_%EC%A0%95%EB%A6%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 연결
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 파일 경로로 이동
%cd /content/drive/MyDrive/rag_intent_chatbot

/content/drive/MyDrive/rag_intent_chatbot


In [ ]:
# 필수 파일 존재 여부 확인

from pathlib import Path

required_files = [
    "models/intent_classifier.pt",
    "data/intents.json",
    "data/documents/sample.txt",
    "src/intent/predict.py",
    "src/rag/document_loader.py",
    "src/rag/chunker.py",
    "src/rag/embedder.py",
    "src/rag/vector_store.py",
    "src/rag/retriever.py",
    "src/chatbot.py",
    "main.py",
]

missing_files = []

for file_path in required_files:
    path = Path(file_path)

    if path.exists():
        print(f"[있음] {file_path}")
    else:
        print(f"[없음] {file_path}")
        missing_files.append(file_path)

print()
print("전체 파일 개수:", len(required_files))
print("없는 파일 개수:", len(missing_files))

if missing_files:
    print("\n없는 파일 목록:")
    for file_path in missing_files:
        print("-", file_path)
else:
    print("\n필수 파일이 모두 존재합니다.")

[있음] models/intent_classifier.pt
[있음] data/intents.json
[있음] data/documents/sample.txt
[있음] src/intent/predict.py
[있음] src/rag/document_loader.py
[있음] src/rag/chunker.py
[있음] src/rag/embedder.py
[있음] src/rag/vector_store.py
[있음] src/rag/retriever.py
[있음] src/chatbot.py
[있음] main.py

전체 파일 개수: 11
없는 파일 개수: 0

필수 파일이 모두 존재합니다.


In [ ]:
# answer_generator.py

%%writefile src/rag/answer_generator.py
from __future__ import annotations

from dataclasses import dataclass

from src.rag.vector_store import SearchResult


MIN_SEARCH_SCORE = 0.30
MAX_CONTEXT_CHUNKS = 3
MAX_CONTEXT_LENGTH = 1500
ANSWER_PREVIEW_LENGTH = 500


@dataclass
class GeneratedAnswer:
    """검색 결과를 이용해 만든 규칙 기반 답변을 저장한다."""

    answer: str
    context: str
    used_results: list[SearchResult]
    sources: list[str]
    has_relevant_context: bool


class AnswerGenerator:
    """Retriever 검색 결과를 규칙 기반 답변으로 정리한다."""

    def __init__(
        self,
        min_search_score: float = MIN_SEARCH_SCORE,
        max_context_chunks: int = MAX_CONTEXT_CHUNKS,
        max_context_length: int = MAX_CONTEXT_LENGTH,
        answer_preview_length: int = ANSWER_PREVIEW_LENGTH,
    ) -> None:
        if not -1.0 <= min_search_score <= 1.0:
            raise ValueError(
                "min_search_score는 -1 이상 1 이하여야 합니다."
            )

        if max_context_chunks < 1:
            raise ValueError(
                "max_context_chunks는 1 이상이어야 합니다."
            )

        if max_context_length < 1:
            raise ValueError(
                "max_context_length는 1 이상이어야 합니다."
            )

        if answer_preview_length < 1:
            raise ValueError(
                "answer_preview_length는 1 이상이어야 합니다."
            )

        self.min_search_score = min_search_score
        self.max_context_chunks = max_context_chunks
        self.max_context_length = max_context_length
        self.answer_preview_length = answer_preview_length

    def generate(
        self,
        question: str,
        search_results: list[SearchResult],
    ) -> GeneratedAnswer:
        """검색 결과를 필터링하고 답변, Context, 출처를 생성한다."""

        if not isinstance(question, str):
            raise TypeError("question은 문자열이어야 합니다.")

        if not isinstance(search_results, list):
            raise TypeError("search_results는 리스트여야 합니다.")

        for result in search_results:
            if not isinstance(result, SearchResult):
                raise TypeError(
                    "search_results에는 SearchResult 객체만 들어갈 수 있습니다."
                )

        if not search_results:
            return GeneratedAnswer(
                answer=(
                    "관련 문서 내용을 찾지 못했습니다.\n"
                    "질문을 조금 더 구체적으로 입력해주세요."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        score_filtered_results = [
            result
            for result in search_results
            if result.score >= self.min_search_score
        ]

        if not score_filtered_results:
            return GeneratedAnswer(
                answer=(
                    "검색된 내용이 질문과 충분히 관련 있다고 "
                    "판단하기 어렵습니다.\n"
                    "다른 표현으로 질문해주세요."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        selected_results: list[SearchResult] = []
        seen_chunk_keys: set[tuple[str, str]] = set()
        seen_texts: set[str] = set()

        for result in score_filtered_results:
            chunk = result.chunk
            cleaned_text = " ".join(chunk.text.split())

            if not cleaned_text:
                continue

            chunk_key = (
                str(chunk.source),
                str(chunk.chunk_id),
            )

            normalized_text = cleaned_text.lower()

            if chunk_key in seen_chunk_keys:
                continue

            if normalized_text in seen_texts:
                continue

            selected_results.append(result)
            seen_chunk_keys.add(chunk_key)
            seen_texts.add(normalized_text)

            if len(selected_results) >= self.max_context_chunks:
                break

        if not selected_results:
            return GeneratedAnswer(
                answer=(
                    "검색 결과는 존재하지만 사용할 수 있는 "
                    "문서 내용이 없습니다."
                ),
                context="",
                used_results=[],
                sources=[],
                has_relevant_context=False,
            )

        context_parts: list[str] = []
        current_context_length = 0

        for result in selected_results:
            cleaned_text = " ".join(result.chunk.text.split())

            context_part = (
                f"[출처: {result.chunk.source}, "
                f"chunk_id: {result.chunk.chunk_id}]\n"
                f"{cleaned_text}"
            )

            remaining_length = (
                self.max_context_length
                - current_context_length
            )

            if remaining_length <= 0:
                break

            if len(context_part) > remaining_length:
                context_part = context_part[:remaining_length].rstrip()

            context_parts.append(context_part)
            current_context_length += len(context_part)

        context = "\n\n".join(context_parts)

        answer_lines = [
            "문서에서 질문과 관련된 내용은 다음과 같습니다.",
            "",
        ]

        for index, result in enumerate(
            selected_results,
            start=1,
        ):
            cleaned_text = " ".join(result.chunk.text.split())
            preview = cleaned_text[:self.answer_preview_length]

            if len(cleaned_text) > self.answer_preview_length:
                preview += "..."

            answer_lines.append(f"{index}. {preview}")

        sources = [
            (
                f"{result.chunk.source}, "
                f"chunk_id={result.chunk.chunk_id}, "
                f"score={result.score:.4f}"
            )
            for result in selected_results
        ]

        answer_lines.extend(
            [
                "",
                "사용한 출처:",
            ]
        )

        for source in sources:
            answer_lines.append(f"- {source}")

        return GeneratedAnswer(
            answer="\n".join(answer_lines),
            context=context,
            used_results=selected_results,
            sources=sources,
            has_relevant_context=True,
        )

Overwriting src/rag/answer_generator.py


In [ ]:
# 파일 만들어졌는지 확인

!sed -n '1,320p' src/rag/answer_generator.py

from __future__ import annotations

from dataclasses import dataclass

from src.rag.vector_store import SearchResult


MIN_SEARCH_SCORE = 0.30
MAX_CONTEXT_CHUNKS = 3
MAX_CONTEXT_LENGTH = 1500
ANSWER_PREVIEW_LENGTH = 500


@dataclass
class GeneratedAnswer:
    """검색 결과를 이용해 만든 규칙 기반 답변을 저장한다."""

    answer: str
    context: str
    used_results: list[SearchResult]
    sources: list[str]
    has_relevant_context: bool


class AnswerGenerator:
    """Retriever 검색 결과를 규칙 기반 답변으로 정리한다."""

    def __init__(
        self,
        min_search_score: float = MIN_SEARCH_SCORE,
        max_context_chunks: int = MAX_CONTEXT_CHUNKS,
        max_context_length: int = MAX_CONTEXT_LENGTH,
        answer_preview_length: int = ANSWER_PREVIEW_LENGTH,
    ) -> None:
        if not -1.0 <= min_search_score <= 1.0:
            raise ValueError(
                "min_search_score는 -1 이상 1 이하여야 합니다."
            )

        if max_context_chunks < 1:
            raise ValueError(
                "max_context_c

In [ ]:
# 테스트

# 검색 결과가 존재하는 경우

from src.rag.answer_generator import AnswerGenerator
from src.rag.chunker import Chunk
from src.rag.vector_store import SearchResult


answer_generator = AnswerGenerator(
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
)

chunk_1 = Chunk(
    text=(
        "임베딩은 텍스트를 숫자로 이루어진 벡터로 변환한 결과이다. "
        "의미가 유사한 텍스트는 벡터 공간에서도 가까운 위치에 놓인다."
    ),
    chunk_id="sample_chunk_0001",
    metadata={"source": "data/documents/sample.txt"},
    source="data/documents/sample.txt",
    start_index=0,
    end_index=70,
)

chunk_2 = Chunk(
    text=(
        "VectorStore는 Chunk와 임베딩을 같은 순서로 저장한다. "
        "질문 임베딩과 문서 임베딩의 코사인 유사도를 계산해 "
        "관련 Chunk를 검색한다."
    ),
    chunk_id="sample_chunk_0002",
    metadata={"source": "data/documents/sample.txt"},
    source="data/documents/sample.txt",
    start_index=60,
    end_index=140,
)

fake_results = [
    SearchResult(
        chunk=chunk_1,
        score=0.82,
        rank=1,
    ),
    SearchResult(
        chunk=chunk_2,
        score=0.71,
        rank=2,
    ),
]

generated_answer = answer_generator.generate(
    question="임베딩과 VectorStore가 무엇인지 알려줘",
    search_results=fake_results,
)

print("answer:")
print(generated_answer.answer)

print("\ncontext:")
print(generated_answer.context)

print("\nhas_relevant_context:")
print(generated_answer.has_relevant_context)

print("\nused_results 개수:")
print(len(generated_answer.used_results))

print("\nsources:")
for source in generated_answer.sources:
    print("-", source)

answer:
문서에서 질문과 관련된 내용은 다음과 같습니다.

1. 임베딩은 텍스트를 숫자로 이루어진 벡터로 변환한 결과이다. 의미가 유사한 텍스트는 벡터 공간에서도 가까운 위치에 놓인다.
2. VectorStore는 Chunk와 임베딩을 같은 순서로 저장한다. 질문 임베딩과 문서 임베딩의 코사인 유사도를 계산해 관련 Chunk를 검색한다.

사용한 출처:
- data/documents/sample.txt, chunk_id=sample_chunk_0001, score=0.8200
- data/documents/sample.txt, chunk_id=sample_chunk_0002, score=0.7100

context:
[출처: data/documents/sample.txt, chunk_id: sample_chunk_0001]
임베딩은 텍스트를 숫자로 이루어진 벡터로 변환한 결과이다. 의미가 유사한 텍스트는 벡터 공간에서도 가까운 위치에 놓인다.

[출처: data/documents/sample.txt, chunk_id: sample_chunk_0002]
VectorStore는 Chunk와 임베딩을 같은 순서로 저장한다. 질문 임베딩과 문서 임베딩의 코사인 유사도를 계산해 관련 Chunk를 검색한다.

has_relevant_context:
True

used_results 개수:
2

sources:
- data/documents/sample.txt, chunk_id=sample_chunk_0001, score=0.8200
- data/documents/sample.txt, chunk_id=sample_chunk_0002, score=0.7100


In [ ]:
# 검색 결과가 존재하지 않는 경우

empty_answer = answer_generator.generate(
    question="문서 내용을 찾아줘",
    search_results=[],
)

print("answer:")
print(empty_answer.answer)

print("\ncontext:")
print(repr(empty_answer.context))

print("\nhas_relevant_context:")
print(empty_answer.has_relevant_context)

print("\nused_results 개수:")
print(len(empty_answer.used_results))

print("\nsources:")
print(empty_answer.sources)

answer:
관련 문서 내용을 찾지 못했습니다.
질문을 조금 더 구체적으로 입력해주세요.

context:
''

has_relevant_context:
False

used_results 개수:
0

sources:
[]


In [ ]:
# 검색 결과는 존재하나 점수가 낮은 경우

low_score_results = [
    SearchResult(
        chunk=chunk_1,
        score=0.18,
        rank=1,
    ),
    SearchResult(
        chunk=chunk_2,
        score=0.09,
        rank=2,
    ),
]

low_score_answer = answer_generator.generate(
    question="자동차 엔진 정비 방법을 알려줘",
    search_results=low_score_results,
)

print("answer:")
print(low_score_answer.answer)

print("\ncontext:")
print(repr(low_score_answer.context))

print("\nhas_relevant_context:")
print(low_score_answer.has_relevant_context)

print("\nused_results 개수:")
print(len(low_score_answer.used_results))

print("\nsources:")
print(low_score_answer.sources)

answer:
검색된 내용이 질문과 충분히 관련 있다고 판단하기 어렵습니다.
다른 표현으로 질문해주세요.

context:
''

has_relevant_context:
False

used_results 개수:
0

sources:
[]


In [ ]:
# 중복 chunk 제거 테스트

duplicate_chunk_id = Chunk(
    text="이 텍스트는 chunk_1과 다른 문장이지만 Chunk ID가 같습니다.",
    chunk_id="sample_chunk_0001",
    metadata={"source": "data/documents/sample.txt"},
    source="data/documents/sample.txt",
    start_index=140,
    end_index=200,
)

duplicate_text_chunk = Chunk(
    text=chunk_2.text,
    chunk_id="sample_chunk_9999",
    metadata={"source": "data/documents/sample.txt"},
    source="data/documents/sample.txt",
    start_index=200,
    end_index=280,
)

duplicate_results = [
    SearchResult(
        chunk=chunk_1,
        score=0.90,
        rank=1,
    ),
    SearchResult(
        chunk=chunk_1,
        score=0.88,
        rank=2,
    ),
    SearchResult(
        chunk=duplicate_chunk_id,
        score=0.80,
        rank=3,
    ),
    SearchResult(
        chunk=chunk_2,
        score=0.75,
        rank=4,
    ),
    SearchResult(
        chunk=duplicate_text_chunk,
        score=0.70,
        rank=5,
    ),
]

duplicate_answer = answer_generator.generate(
    question="임베딩과 VectorStore를 설명해줘",
    search_results=duplicate_results,
)

print("answer:")
print(duplicate_answer.answer)

print("\ncontext:")
print(duplicate_answer.context)

print("\nhas_relevant_context:")
print(duplicate_answer.has_relevant_context)

print("\nused_results 개수:")
print(len(duplicate_answer.used_results))

print("\n실제로 사용된 Chunk:")
for result in duplicate_answer.used_results:
    print(
        result.chunk.chunk_id,
        result.score,
        result.chunk.text[:50],
    )

print("\nsources:")
for source in duplicate_answer.sources:
    print("-", source)

answer:
문서에서 질문과 관련된 내용은 다음과 같습니다.

1. 임베딩은 텍스트를 숫자로 이루어진 벡터로 변환한 결과이다. 의미가 유사한 텍스트는 벡터 공간에서도 가까운 위치에 놓인다.
2. VectorStore는 Chunk와 임베딩을 같은 순서로 저장한다. 질문 임베딩과 문서 임베딩의 코사인 유사도를 계산해 관련 Chunk를 검색한다.

사용한 출처:
- data/documents/sample.txt, chunk_id=sample_chunk_0001, score=0.9000
- data/documents/sample.txt, chunk_id=sample_chunk_0002, score=0.7500

context:
[출처: data/documents/sample.txt, chunk_id: sample_chunk_0001]
임베딩은 텍스트를 숫자로 이루어진 벡터로 변환한 결과이다. 의미가 유사한 텍스트는 벡터 공간에서도 가까운 위치에 놓인다.

[출처: data/documents/sample.txt, chunk_id: sample_chunk_0002]
VectorStore는 Chunk와 임베딩을 같은 순서로 저장한다. 질문 임베딩과 문서 임베딩의 코사인 유사도를 계산해 관련 Chunk를 검색한다.

has_relevant_context:
True

used_results 개수:
2

실제로 사용된 Chunk:
sample_chunk_0001 0.9 임베딩은 텍스트를 숫자로 이루어진 벡터로 변환한 결과이다. 의미가 유사한 텍스트는 벡터 공
sample_chunk_0002 0.75 VectorStore는 Chunk와 임베딩을 같은 순서로 저장한다. 질문 임베딩과 문서 임

sources:
- data/documents/sample.txt, chunk_id=sample_chunk_0001, score=0.9000
- data/documents/sample.txt, chunk_id=sample_chunk_0002, score=0.7500


In [ ]:
# 잘못된 입력의 경우

try:
    answer_generator.generate(
        question="테스트 질문",
        search_results=["SearchResult가 아닌 문자열"],
    )
except Exception as error:
    print(type(error).__name__)
    print(error)

TypeError
search_results에는 SearchResult 객체만 들어갈 수 있습니다.


In [ ]:
# Retriever 결과 기반 테스트

from main import build_retriever
from src.rag.answer_generator import AnswerGenerator


retriever = build_retriever()

answer_generator = AnswerGenerator(
    min_search_score=0.30,
    max_context_chunks=3,
    max_context_length=1500,
    answer_preview_length=500,
)

test_questions = [
    "임베딩이 무엇인지 문서에서 찾아줘",
    "Chunk를 사용하는 이유를 문서에서 찾아줘",
    "VectorStore는 어떤 역할을 하는지 찾아줘",
    "RAG와 관련 없는 자동차 엔진 정비 방법을 찾아줘",
]

for question in test_questions:
    print("=" * 80)
    print("사용자 질문:", question)

    search_results = retriever.retrieve(
        query=question,
        top_k=3,
    )

    print("검색 결과 개수:", len(search_results))

    for result in search_results:
        print(
            f"rank={result.rank}, "
            f"score={result.score:.4f}, "
            f"chunk_id={result.chunk.chunk_id}"
        )

    generated_answer = answer_generator.generate(
        question=question,
        search_results=search_results,
    )

    print()
    print("생성된 answer:")
    print(generated_answer.answer)

    print()
    print("사용한 Chunk 개수:", len(generated_answer.used_results))
    print(
        "has_relevant_context:",
        generated_answer.has_relevant_context,
    )

    print("출처:")
    for source in generated_answer.sources:
        print("-", source)

    print()

[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료
사용자 질문: 임베딩이 무엇인지 문서에서 찾아줘
검색 결과 개수: 3
rank=1, score=0.2757, chunk_id=sample_chunk_0003
rank=2, score=0.2729, chunk_id=sample_chunk_0008
rank=3, score=0.2146, chunk_id=sample_chunk_0001

생성된 answer:
검색된 내용이 질문과 충분히 관련 있다고 판단하기 어렵습니다.
다른 표현으로 질문해주세요.

사용한 Chunk 개수: 0
has_relevant_context: False
출처:

사용자 질문: Chunk를 사용하는 이유를 문서에서 찾아줘
검색 결과 개수: 3
rank=1, score=0.2746, chunk_id=sample_chunk_0003
rank=2, score=0.2193, chunk_id=sample_chunk_0008
rank=3, score=0.2117, chunk_id=sample_chunk_0001

생성된 answer:
검색된 내용이 질문과 충분히 관련 있다고 판단하기 어렵습니다.
다른 표현으로 질문해주세요.

사용한 Chunk 개수: 0
has_relevant_context: False
출처:

사용자 질문: VectorStore는 어떤 역할을 하는지 찾아줘
검색 결과 개수: 3
rank=1, score=0.2875, chunk_id=sample_chunk_0008
rank=2, score=0.2636, chunk_id=sample_chunk_0000
rank=3, score=0.2596, chunk_id=sample_chunk_0005

생성된 answer:
검색된 내용이 질문과 충분히 관련 있다고 판단하기 어렵습니다.
다른 표현으로 질문해주세요.

사용한 Chunk 개수: 0
has_relevant_context: False
출

In [ ]:
# chatbot.py 수정

'''
기존 chatbot.py의 경우 검색 결과를 그대로 문자열로 바꾸었었다(원본 데이터의 형식만 가공).
해당하는 부분을 AnswerGenerator가 담당하는 형식으로 바꾸었다.
이를 통해 점수 필터링, 중복 제거, 길이 제한, 출처 표기, 예외 처리의 기능이 추가되었다.
AnswerGenerator는 기존에 만든 class로 검색 결과를 필터링, 가공하는 규칙 기반 엔진임
'''


'\n기존 chatbot.py의 경우 검색 결과를 그대로 문자열로 바꾸었었다(원본 데이터의 형식만 가공).\n해당하는 부분을 AnswerGenerator가 담당하는 형식으로 바꾸었다.\n이를 통해 점수 필터링, 중복 제거, 길이 제한, 출처 표기, 예외 처리의 기능이 추가되었다.\nAnswerGenerator는 기존에 만든 class로 검색 결과를 필터링, 가공하는 규칙 기반 엔진임\n'

In [ ]:
%%writefile src/chatbot.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from src.rag.answer_generator import AnswerGenerator
from src.rag.retriever import Retriever
from src.rag.vector_store import SearchResult


@dataclass
class ChatbotResult:
    """사용자 입력 한 건의 처리 결과를 저장한다."""

    user_input: str
    predicted_intent: str
    confidence: float
    is_fallback: bool
    requires_rag: bool
    response: str
    search_results: list[SearchResult]


class Chatbot:
    """Intent 예측 결과에 따라 일반 응답과 RAG 검색을 분기한다."""

    def __init__(
        self,
        intent_predictor: Any,
        retriever: Retriever,
        answer_generator: AnswerGenerator, # | None = None,(LLM 기반으로 확장)
        confidence_threshold: float = 0.60,
        retrieval_top_k: int = 3,
        preview_length: int = 200,
    ) -> None:
        if not 0.0 <= confidence_threshold <= 1.0:
            raise ValueError(
                "confidence_threshold는 0 이상 1 이하여야 합니다."
            )

        if retrieval_top_k < 1:
            raise ValueError(
                "retrieval_top_k는 1 이상이어야 합니다."
            )

        if preview_length < 1:
            raise ValueError(
                "preview_length는 1 이상이어야 합니다."
            )

        if not isinstance(answer_generator, AnswerGenerator):
            raise TypeError(
                "answer_generator는 AnswerGenerator 객체여야 합니다."
            )

        self.intent_predictor = intent_predictor
        self.retriever = retriever
        self.answer_generator = answer_generator
        self.confidence_threshold = confidence_threshold
        self.retrieval_top_k = retrieval_top_k
        self.preview_length = preview_length

    def process_message(
        self,
        user_input: str,
    ) -> ChatbotResult:
        """사용자 문장을 분류하고 알맞은 처리 경로로 전달한다."""

        if not isinstance(user_input, str):
            raise TypeError("user_input은 문자열이어야 합니다.")

        user_input = user_input.strip()

        if not user_input:
            return ChatbotResult(
                user_input=user_input,
                predicted_intent="fallback",
                confidence=0.0,
                is_fallback=True,
                requires_rag=False,
                response="질문을 입력해주세요.",
                search_results=[],
            )

        prediction = self.intent_predictor.predict(
            text=user_input,
            threshold=self.confidence_threshold,
            top_k=3,
        )

        predicted_intent = prediction["intent"]
        confidence = float(prediction["confidence"])
        is_fallback = predicted_intent == "fallback"
        requires_rag = bool(prediction["requires_rag"])

        if is_fallback:
            return ChatbotResult(
                user_input=user_input,
                predicted_intent=predicted_intent,
                confidence=confidence,
                is_fallback=True,
                requires_rag=False,
                response=prediction["response"],
                search_results=[],
            )

        if not requires_rag:
            return ChatbotResult(
                user_input=user_input,
                predicted_intent=predicted_intent,
                confidence=confidence,
                is_fallback=False,
                requires_rag=False,
                response=prediction["response"],
                search_results=[],
            )

        search_results = self.retriever.retrieve(
            query=user_input,
            top_k=self.retrieval_top_k,
        )

        generated_answer = self.answer_generator.generate(
            question=user_input,
            search_results=search_results,
        )

        return ChatbotResult(
            user_input=user_input,
            predicted_intent=predicted_intent,
            confidence=confidence,
            is_fallback=False,
            requires_rag=True,
            response=generated_answer.answer,
            search_results=search_results,
        )



Overwriting src/chatbot.py


In [ ]:
# 파일 확인
!sed -n '1,320p' src/chatbot.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from src.rag.answer_generator import AnswerGenerator
from src.rag.retriever import Retriever
from src.rag.vector_store import SearchResult


@dataclass
class ChatbotResult:
    """사용자 입력 한 건의 처리 결과를 저장한다."""

    user_input: str
    predicted_intent: str
    confidence: float
    is_fallback: bool
    requires_rag: bool
    response: str
    search_results: list[SearchResult]


class Chatbot:
    """Intent 예측 결과에 따라 일반 응답과 RAG 검색을 분기한다."""

    def __init__(
        self,
        intent_predictor: Any,
        retriever: Retriever,
        answer_generator: AnswerGenerator, # | None = None,(LLM 기반으로 확장)
        confidence_threshold: float = 0.60,
        retrieval_top_k: int = 3,
        preview_length: int = 200,
    ) -> None:
        if not 0.0 <= confidence_threshold <= 1.0:
            raise ValueError(
                "confidence_threshold는 0 이상 1 이하여야 합니다."
            )

        if ret

In [ ]:
# main.py 코드 변경
'''
변경점
1) AnswerGenerator import
2) 답변 생성 설정값 추가
3) build_chatbot() 안에서 AnswerGenerator 한번만 생성
RAG 챗봇과 Intent Classifier를 통합 실행한다.'''

'\n변경점\n1) AnswerGenerator import\n2) 답변 생성 설정값 추가\n3) build_chatbot() 안에서 AnswerGenerator 한번만 생성\nRAG 챗봇과 Intent Classifier를 통합 실행한다.'

In [ ]:
%%writefile main.py
"""RAG 챗봇과 Intent Classifier를 통합 실행한다."""

from pathlib import Path
import os # LLM 기반 확장

from google import genai # LLM 기반 확장
from src.intent.predict import IntentPredictor
from src.rag.document_loader import Document, load_documents
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_documents
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever
from src.rag.answer_generator import AnswerGenerator
from src.chatbot import Chatbot, ChatbotResult


MODEL_PATH = "models/intent_classifier.pt"
DOCUMENT_DIRECTORY = "data/documents"

CONFIDENCE_THRESHOLD = 0.60
RETRIEVAL_TOP_K = 3

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
PREVIEW_LENGTH = 200

MIN_SEARCH_SCORE = 0.30
MAX_CONTEXT_CHUNKS = 3
MAX_CONTEXT_LENGTH = 1500
ANSWER_PREVIEW_LENGTH = 500

# LLM 기반 확장
GEMINI_MODEL_NAME = "gemini-3.5-flash-lite"
GEMINI_SECRET_NAME = "GEMINI_API_KEY"
GEMINI_TEMPERATURE = 0.20
GEMINI_MAX_OUTPUT_TOKENS = 500

EXIT_COMMANDS = {"exit", "quit", "종료"}


def build_retriever() -> Retriever:
    """문서를 처리하고 검색에 사용할 Retriever를 생성한다."""

    document_directory = Path(DOCUMENT_DIRECTORY)

    if not document_directory.exists():
        raise FileNotFoundError(
            f"문서 디렉터리를 찾을 수 없습니다: {DOCUMENT_DIRECTORY}"
        )

    text_files = list(document_directory.glob("*.txt"))

    if not text_files:
        raise FileNotFoundError(
            f"불러올 .txt 문서가 없습니다: {DOCUMENT_DIRECTORY}"
        )

    print("[1/6] 문서를 불러오는 중...")

    documents = load_documents(
        directory_path=DOCUMENT_DIRECTORY,
    )

    if not documents:
        raise RuntimeError("문서를 불러왔지만 문서 목록이 비어 있습니다.")

    print(f"      불러온 문서 수: {len(documents)}")

    print("[2/6] 문서를 전처리하는 중...")

    preprocessed_documents: list[Document] = []

    for document in documents:
        preprocessed_document = Document(
            text=preprocess_text(document.text),
            metadata=document.metadata,
        )

        preprocessed_documents.append(preprocessed_document)

    print("[3/6] 문서를 Chunk로 나누는 중...")

    chunks = chunk_documents(
        documents=preprocessed_documents,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    if not chunks:
        raise RuntimeError("문서에서 Chunk가 생성되지 않았습니다.")

    print(f"      생성된 Chunk 수: {len(chunks)}")

    print("[4/6] TextEmbedder를 불러오는 중...")

    try:
        embedder = TextEmbedder()
    except Exception as error:
        raise RuntimeError(
            "Sentence Transformer 모델을 불러오지 못했습니다."
        ) from error

    print("[5/6] Chunk 임베딩을 생성하는 중...")

    chunk_texts = [
        chunk.text
        for chunk in chunks
    ]

    chunk_embeddings = embedder.encode_texts(
        texts=chunk_texts,
    )

    print("[6/6] VectorStore와 Retriever를 생성하는 중...")

    try:
        vector_store = VectorStore()

        vector_store.add(
            chunks=chunks,
            embeddings=chunk_embeddings,
        )
    except Exception as error:
        raise RuntimeError(
            "VectorStore 생성 또는 데이터 저장에 실패했습니다."
        ) from error

    retriever = Retriever(
        embedder=embedder,
        vector_store=vector_store,
    )

    print("      Retriever 생성 완료")

    return retriever


def build_chatbot() -> Chatbot:
    """IntentPredictor, Retriever, AnswerGenerator를 생성한다."""

    model_path = Path(MODEL_PATH)

    if not model_path.exists():
        raise FileNotFoundError(
            f"학습된 Intent 모델을 찾을 수 없습니다: {MODEL_PATH}"
        )

    print("IntentPredictor를 불러오는 중...")

    intent_predictor = IntentPredictor(
        model_path=MODEL_PATH,
    )

    retriever = build_retriever()

    # LLM 확장으로 인한 수정
    print("Gemini API 키를 확인하는 중...")

    api_key = load_gemini_api_key()

    print("Gemini 클라이언트를 생성하는 중...")

    gemini_client = genai.Client(
        api_key=api_key,
    )

    answer_generator = AnswerGenerator(
        client=gemini_client, # LLM 확장으로 인한 수정
        model_name=GEMINI_MODEL_NAME, # LLM 확장으로 인한 수정
        min_search_score=MIN_SEARCH_SCORE,
        max_context_chunks=MAX_CONTEXT_CHUNKS,
        max_context_length=MAX_CONTEXT_LENGTH,
        answer_preview_length=ANSWER_PREVIEW_LENGTH,
        temperature=GEMINI_TEMPERATURE, # LLM 확장으로 인한 수정
        max_output_tokens=GEMINI_MAX_OUTPUT_TOKENS, # LLM 확장으로 인한 수정
    )

    chatbot = Chatbot(
        intent_predictor=intent_predictor,
        retriever=retriever,
        answer_generator=answer_generator,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        retrieval_top_k=RETRIEVAL_TOP_K,
        preview_length=PREVIEW_LENGTH,
    )
    # LLM 기반 확장

    api_key = load_gemini_api_key()

    gemini_client = genai.Client(
        api_key=api_key,
    )

    return chatbot


def print_result(result: ChatbotResult) -> None:
    """Chatbot 처리 결과를 읽기 쉬운 형태로 출력한다."""

    print()
    print("-" * 70)
    print(f"예측 Intent : {result.predicted_intent}")
    print(f"Confidence  : {result.confidence:.4f}")
    print(f"Fallback    : {result.is_fallback}")
    print(f"RAG 사용    : {result.requires_rag}")
    print(f"검색 결과 수: {len(result.search_results)}")
    print()
    print(f"챗봇: {result.response}")

    if result.search_results:
        print()
        print("[Retriever 원본 검색 결과]")

        for search_result in result.search_results:
            chunk = search_result.chunk

            preview = chunk.text[:PREVIEW_LENGTH]

            if len(chunk.text) > PREVIEW_LENGTH:
                preview += "..."

            print()
            print(
                f"{search_result.rank}위 | "
                f"score={search_result.score:.4f}"
            )
            print(f"source   : {chunk.source}")
            print(f"chunk_id : {chunk.chunk_id}")
            print(f"text     : {preview}")

    print("-" * 70)


def main() -> None:
    """챗봇을 초기화하고 사용자 입력을 반복해서 처리한다."""

    print("=" * 70)
    print("RAG 챗봇 + PyTorch Intent Classifier")
    print("=" * 70)

    try:
        chatbot = build_chatbot()

        print()
        print("챗봇 초기화가 완료되었습니다.")
        print("질문을 입력해주세요.")
        print("종료 명령: exit, quit, 종료")
        print()

        while True:
            user_input = input("사용자: ").strip()

            if user_input.lower() in EXIT_COMMANDS:
                print("챗봇을 종료합니다.")
                break

            if not user_input:
                print("챗봇: 질문을 입력해주세요.")
                continue

            result = chatbot.process_message(
                user_input=user_input,
            )

            print_result(result)

    except KeyboardInterrupt:
        print()
        print("사용자 요청으로 챗봇을 종료합니다.")

    except FileNotFoundError as error:
        print()
        print(f"[파일 오류] {error}")

    except RuntimeError as error:
        print()
        print(f"[초기화 오류] {error}")

    except Exception as error:
        print()
        print(f"[예상하지 못한 오류] {error}")

# LLM 기반 확장

def load_gemini_api_key() -> str:
    environment_key = os.getenv(GEMINI_SECRET_NAME)

    if environment_key:
        return environment_key

    try:
        from google.colab import userdata

        api_key = userdata.get(GEMINI_SECRET_NAME)

        if api_key:
            return api_key

    except ImportError:
        pass

    except Exception as error:
        raise RuntimeError(
            "Colab Secrets에서 GEMINI_API_KEY를 읽지 못했습니다. "
            "Notebook access 설정을 확인해주세요."
        ) from error

    raise RuntimeError(
        "Gemini API 키가 없습니다. "
        "Colab Secrets에 GEMINI_API_KEY를 등록해주세요."
    )

if __name__ == "__main__":
    main()

Overwriting main.py


In [ ]:
!sed -n '1,400p' main.py

"""RAG 챗봇과 Intent Classifier를 통합 실행한다."""

from pathlib import Path
import os # LLM 기반 확장

from google import genai # LLM 기반 확장
from src.intent.predict import IntentPredictor
from src.rag.document_loader import Document, load_documents
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_documents
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever
from src.rag.answer_generator import AnswerGenerator
from src.chatbot import Chatbot, ChatbotResult


MODEL_PATH = "models/intent_classifier.pt"
DOCUMENT_DIRECTORY = "data/documents"

CONFIDENCE_THRESHOLD = 0.60
RETRIEVAL_TOP_K = 3

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
PREVIEW_LENGTH = 200

MIN_SEARCH_SCORE = 0.30
MAX_CONTEXT_CHUNKS = 3
MAX_CONTEXT_LENGTH = 1500
ANSWER_PREVIEW_LENGTH = 500

# LLM 기반 확장
GEMINI_MODEL_NAME = "gemini-2.5-flash-lite"
GEMINI_SECRET_NAME = "GEMINI_API_KEY"
GEMINI_TEMPERATURE = 0.20
GEMINI_MAX_OUTPUT_TOKENS 

In [ ]:
# 전체 Chatbot 생성 테스트

from main import build_chatbot

chatbot = build_chatbot()

print()
print("Chatbot 생성 완료")
print("Chatbot 타입:", type(chatbot).__name__)
print(
    "AnswerGenerator 타입:",
    type(chatbot.answer_generator).__name__,
)

IntentPredictor를 불러오는 중...
[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료

Chatbot 생성 완료
Chatbot 타입: Chatbot
AnswerGenerator 타입: AnswerGenerator


In [ ]:
# 일반 Intent 회귀 테스트
from main import build_chatbot

chatbot = build_chatbot()

print()
print("Chatbot 생성 완료")
print("Chatbot 타입:", type(chatbot).__name__)
print(
    "AnswerGenerator 타입:",
    type(chatbot.answer_generator).__name__,
)

IntentPredictor를 불러오는 중...
[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료

Chatbot 생성 완료
Chatbot 타입: Chatbot
AnswerGenerator 타입: AnswerGenerator


In [ ]:
# fallback 회귀 테스트

fallback_input = "나는 짱이야"

fallback_result = chatbot.process_message(
    user_input=fallback_input,
)

print("사용자 입력:", fallback_result.user_input)
print("predicted_intent:", fallback_result.predicted_intent)
print(f"confidence: {fallback_result.confidence:.4f}")
print("fallback 여부:", fallback_result.is_fallback)
print("requires_rag:", fallback_result.requires_rag)
print("검색 결과 개수:", len(fallback_result.search_results))
print("최종 response:")
print(fallback_result.response)

사용자 입력: 나는 짱이야
predicted_intent: fallback
confidence: 0.2976
fallback 여부: True
requires_rag: False
검색 결과 개수: 0
최종 response:
질문의 의도를 확실하게 판단하지 못했습니다. 조금 더 구체적으로 질문해주세요.


In [ ]:
# document_query 통합 테스트

document_inputs = [
    "문서에서 임베딩이 무엇인지 찾아줘",
    "문서에서 Chunk를 사용하는 이유를 찾아줘",
    "자동차 엔진 오일 교환법을 문서에서 찾아줘",
]

for user_input in document_inputs:
    result = chatbot.process_message(
        user_input=user_input,
    )

    print("=" * 80)
    print("사용자 입력:", result.user_input)
    print("predicted_intent:", result.predicted_intent)
    print(f"confidence: {result.confidence:.4f}")
    print("fallback 여부:", result.is_fallback)
    print("requires_rag:", result.requires_rag)
    print("검색 결과 개수:", len(result.search_results))

    print()
    print("최종 response:")
    print(result.response)

    if result.search_results:
        print()
        print("Retriever 검색 결과:")

        for search_result in result.search_results:
            print(
                f"- rank={search_result.rank}, "
                f"score={search_result.score:.4f}, "
                f"chunk_id={search_result.chunk.chunk_id}"
            )

    print()

사용자 입력: 문서에서 임베딩이 무엇인지 찾아줘
predicted_intent: document_query
confidence: 0.9933
fallback 여부: False
requires_rag: True
검색 결과 개수: 3

최종 response:
문서에서 질문과 관련된 내용은 다음과 같습니다.

1. 프롬프트에 넣어 근거 중심의 응답을 만들 수 있다. 현재 단계에서는 이 전체 구조 중 가장 앞부분인 문서 로딩, 공백 및 줄바꿈 전처리, 문단과 문장을 고려한 청킹을 구현한다. 이후에는 임베딩 생성, 벡터 저장소, Retriever, Intent Classifier와 RAG 라우팅, 최종 챗봇 인터페이스 순서로 확장할 예정이다.

사용한 출처:
- sample.txt, chunk_id=sample_chunk_0008, score=0.3169

Retriever 검색 결과:
- rank=1, score=0.3169, chunk_id=sample_chunk_0008
- rank=2, score=0.2900, chunk_id=sample_chunk_0003
- rank=3, score=0.2864, chunk_id=sample_chunk_0005

사용자 입력: 문서에서 Chunk를 사용하는 이유를 찾아줘
predicted_intent: document_query
confidence: 0.9933
fallback 여부: False
requires_rag: True
검색 결과 개수: 3

최종 response:
문서에서 질문과 관련된 내용은 다음과 같습니다.

1. 너무 긴 문단이나 문장은 최종적으로 글자 수 기준으로 나눌 수 있으며, 인접 Chunk 사이에 일부 내용을 겹치게 두면 경계 부근의 문맥이 사라지는 문제를 완화할 수 있다. 임베딩의 의미 컴퓨터는 문장의 의미를 사람처럼 직접 이해하지 못하므로, 텍스트를 수치 벡터로 변환하는 과정이 필요하다. 임베딩은 단어, 문장, 문서 조각의 의미적 특징을 여러 차원의 숫자로 표현한 벡터이다. 의미가 비슷한 문장들은

In [ ]:
# 전체 통합 테스트

integration_test_inputs = [
    "안녕하세요",
    "고마워요",
    "너는 누구야?",
    "사용 방법을 알려줘",
    "문서에서 임베딩이 무엇인지 찾아줘",
    "문서에서 Chunk를 사용하는 이유를 찾아줘",
    "자동차 엔진 오일 교환법을 문서에서 찾아줘",
    "파란 생각이 조용하게 숫자를 걸어간다",
]

for user_input in integration_test_inputs:
    result = chatbot.process_message(
        user_input=user_input,
    )

    print("=" * 80)
    print("사용자 입력:", result.user_input)
    print("predicted_intent:", result.predicted_intent)
    print(f"confidence: {result.confidence:.4f}")
    print("fallback 여부:", result.is_fallback)
    print("requires_rag:", result.requires_rag)
    print("검색 결과 개수:", len(result.search_results))
    print()
    print("최종 response:")
    print(result.response)
    print()

사용자 입력: 안녕하세요
predicted_intent: greeting
confidence: 0.9207
fallback 여부: False
requires_rag: False
검색 결과 개수: 0

최종 response:
안녕하세요. 무엇을 도와드릴까요?

사용자 입력: 고마워요
predicted_intent: fallback
confidence: 0.2976
fallback 여부: True
requires_rag: False
검색 결과 개수: 0

최종 response:
질문의 의도를 확실하게 판단하지 못했습니다. 조금 더 구체적으로 질문해주세요.

사용자 입력: 너는 누구야?
predicted_intent: bot_info
confidence: 0.9749
fallback 여부: False
requires_rag: False
검색 결과 개수: 0

최종 response:
사용자의 의도를 분류하고, 문서 질문은 RAG 검색기로 전달하는 챗봇입니다.

사용자 입력: 사용 방법을 알려줘
predicted_intent: help
confidence: 0.9074
fallback 여부: False
requires_rag: False
검색 결과 개수: 0

최종 response:
현재는 인텐트 분류 기능을 제공하며, 다음 단계에서 RAG 문서 검색 기능이 연결됩니다.

사용자 입력: 문서에서 임베딩이 무엇인지 찾아줘
predicted_intent: document_query
confidence: 0.9933
fallback 여부: False
requires_rag: True
검색 결과 개수: 3

최종 response:
문서에서 질문과 관련된 내용은 다음과 같습니다.

1. 프롬프트에 넣어 근거 중심의 응답을 만들 수 있다. 현재 단계에서는 이 전체 구조 중 가장 앞부분인 문서 로딩, 공백 및 줄바꿈 전처리, 문단과 문장을 고려한 청킹을 구현한다. 이후에는 임베딩 생성, 벡터 저장소, Retriever, Intent Classifier와 RAG 라우팅, 최종 챗봇 